In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path

# ---------------------------
# Paper-ready / LaTeX-friendly style
# ---------------------------
BASE_FONT_SIZE = 18
AXIS_LABEL_SIZE = 22
AXIS_TITLE_SIZE = 20
TICK_LABEL_SIZE = 16
LEGEND_FONT_SIZE = 15
VALUE_LABEL_SIZE = 14

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": BASE_FONT_SIZE,
    "axes.titlesize": AXIS_TITLE_SIZE,
    "axes.labelsize": AXIS_LABEL_SIZE,
    "xtick.labelsize": TICK_LABEL_SIZE,
    "ytick.labelsize": TICK_LABEL_SIZE,
    "legend.fontsize": LEGEND_FONT_SIZE,
    "legend.title_fontsize": LEGEND_FONT_SIZE,
    "axes.linewidth": 1.0,
    "grid.linewidth": 0.6,
    "hatch.linewidth": 0.9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ---------------------------
# Paths
# ---------------------------

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "results" / "cifar100" / "cifar100_segmented.csv").exists() and (candidate / "plots").exists():
            return candidate
    raise FileNotFoundError("Could not locate the XAIV project root from the current working directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
PLOT_DIR = PROJECT_ROOT / "plots" / "5.4.2_ratio"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CIFAR_CSV = PROJECT_ROOT / "results" / "cifar100" / "cifar100_segmented.csv"
IMAGENET_CSV = PROJECT_ROOT / "results" / "vggnet16" / "ALL_vggnet.csv"

# ---------------------------
# Result canonicalization
# ---------------------------
def canon_result(x):
    s = str(x).lower()
    if "unsat" in s:
        return "Safe"
    if "sat" in s:
        return "Unsafe"
    return "Unknown"

ORDER = ["Safe", "Unknown", "Unsafe"]
OUTCOME_STYLE = {
    "Safe": {"facecolor": "#111111", "hatch": ""},
    "Unknown": {"facecolor": "white", "hatch": "///"},
    "Unsafe": {"facecolor": "#B22222", "hatch": "xx"},
}
TEXT_COLOR = {"Safe": "white", "Unknown": "black", "Unsafe": "white"}
BASELINE_LABEL = r"$\alpha\beta$-CROWN"
CSI_LABEL = r"$\alpha\beta$-CROWN w/ CSI [object]"

def _make_bins(n_bins):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    labels = [f"{edges[i]:.1f}–{edges[i+1]:.1f}" for i in range(n_bins)]
    return edges, labels

def _stacked_props(series_bins, series_result, bin_labels):
    counts = (
        pd.crosstab(series_bins, series_result)
        .reindex(bin_labels)
        .fillna(0)
    )
    n = counts.sum(axis=1).astype(int)
    props = counts.div(n.replace(0, np.nan), axis=0).fillna(0)
    return props, n

def compute_semantic_vs_global_by_obj_ratio(
    df,
    *,
    key_global,
    key_obj,
    tag_obj="fix_nonmask",
    tag_global="global",
    n_bins=10,
    require_total=None,
    require_full_k=True
):
    d = df.copy()

    # numeric cleanup
    for c in ["eps", "k", "total", "segment_index", "num_changed"]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    for c in ["tag", "model", "image"]:
        if c in d.columns:
            d[c] = d[c].astype(str).str.strip()

    d = d.dropna(subset=["total", "k"]).copy()

    if require_total is not None:
        d = d[d["total"].astype(int) == int(require_total)].copy()

    if require_full_k:
        d["full_k"] = (d["total"] // 3).astype(int)
        d = d[d["k"].astype(int) == d["full_k"]].copy()

    d = d[d["tag"].isin([tag_obj, tag_global])].copy()
    if d.empty:
        raise ValueError("After filtering tags/total/k, dataframe is empty.")

    d["result_cat"] = d["result"].apply(canon_result)

    obj = d[d["tag"] == tag_obj][key_obj + ["num_changed", "result_cat"]].copy()
    glb = d[d["tag"] == tag_global][key_global + ["result_cat"]].copy()

    if obj.empty:
        raise ValueError(f"No rows found for object tag='{tag_obj}'.")
    if glb.empty:
        raise ValueError(
            f"No rows found for global tag='{tag_global}'. "
            f"(Your CSV may not contain 'global' runs.)"
        )

    obj = obj.rename(columns={"num_changed": "obj_changed", "result_cat": "sem_result"})
    glb = glb.rename(columns={"result_cat": "glb_result"}).drop_duplicates(subset=key_global, keep="first")

    paired = obj.merge(glb, on=key_global, how="left")
    if paired["glb_result"].isna().any():
        missing = int(paired["glb_result"].isna().sum())
        print(f"[WARN] Missing global result for {missing} object rows (after merge). Dropping those rows.")
        paired = paired.dropna(subset=["glb_result"]).copy()

    edges, bin_labels = _make_bins(n_bins)

    paired["obj_ratio"] = paired["obj_changed"] / paired["total"]
    paired["obj_bin"] = pd.cut(paired["obj_ratio"], bins=edges, labels=bin_labels, include_lowest=True)

    props_sem, n_per_bin = _stacked_props(paired["obj_bin"], paired["sem_result"], bin_labels)
    props_glb, _         = _stacked_props(paired["obj_bin"], paired["glb_result"], bin_labels)

    return props_sem, props_glb, n_per_bin, bin_labels

def draw_grouped_stacked_sem_vs_global(
    ax,
    props_sem,
    props_glb,
    n_per_bin,
    xlabels,
    *,
    xlabel="Object ratio",
    title="",
    width=0.40,
    gap=0.10,
    hatch_glb="///",
    show_pct_on_sem=True,
    pct_thresh=0.06,       # lower threshold so more labels appear (still readable)
    show_n=True
):
    x = np.arange(len(xlabels))
    xS = x - (width/2 + gap/2)  # Object-aware
    xG = x + (width/2 + gap/2)  # Baseline

    bottomS = np.zeros(len(xlabels))
    bottomG = np.zeros(len(xlabels))

    for lab in ORDER:
        valsS = props_sem.get(lab, pd.Series(0, index=xlabels)).reindex(xlabels).fillna(0).values
        valsG = props_glb.get(lab, pd.Series(0, index=xlabels)).reindex(xlabels).fillna(0).values
        style = OUTCOME_STYLE[lab]

        ax.bar(
            xS,
            valsS,
            bottom=bottomS,
            width=width,
            facecolor=style["facecolor"],
            edgecolor="black",
            linewidth=0.8,
            hatch=style["hatch"],
        )
        ax.bar(
            xG,
            valsG,
            bottom=bottomG,
            width=width,
            facecolor=style["facecolor"],
            edgecolor="black",
            linewidth=0.8,
            hatch=style["hatch"] + hatch_glb,
        )

        if show_pct_on_sem:
            for j, v in enumerate(valsS):
                if v >= pct_thresh:
                    ax.text(
                        xS[j],
                        bottomS[j] + v / 2,
                        f"{v*100:.0f}%",
                        ha="center",
                        va="center",
                        fontsize=VALUE_LABEL_SIZE,
                        color=TEXT_COLOR[lab],
                    )

        bottomS += valsS
        bottomG += valsG

    if show_n:
        n_vals = n_per_bin.reindex(xlabels).fillna(0).astype(int).values
        for j, n in enumerate(n_vals):
            if n > 0:
                ax.text(
                    x[j],
                    1.03,
                    f"n={n}",
                    ha="center",
                    va="bottom",
                    fontsize=12
                )

    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, rotation=30, ha="right")
    ax.set_title(title, pad=12)
    ax.set_xlabel(xlabel)
    ax.set_ylim(0, 1.12)

    ax.grid(axis="y", linestyle="--", alpha=0.30)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# ============================================================
# Load both datasets
# ============================================================
df_cifar = pd.read_csv(CIFAR_CSV)
df_imnet = pd.read_csv(IMAGENET_CSV)

# ------------------------------------------------------------
# CIFAR-100
# expects tag in {fix_nonmask, global} for semantic vs global
# ------------------------------------------------------------
KEY_G_CIFAR = [c for c in ["model", "image", "eps", "k", "total"] if c in df_cifar.columns]
KEY_O_CIFAR = [c for c in ["model", "image", "segment_index", "eps", "k", "total"] if c in df_cifar.columns]

props_c_sem, props_c_glb, n_c, labels_c = compute_semantic_vs_global_by_obj_ratio(
    df_cifar,
    key_global=KEY_G_CIFAR,
    key_obj=KEY_O_CIFAR,
    tag_obj="fix_nonmask",
    tag_global="global",
    n_bins=5,
    require_total=None,
    require_full_k=True
)

# ------------------------------------------------------------
# ImageNet
# ------------------------------------------------------------
KEY_G_IM = [c for c in ["image", "eps", "k", "total"] if c in df_imnet.columns]
KEY_O_IM = [c for c in ["image", "segment_index", "eps", "k", "total"] if c in df_imnet.columns]

props_i_sem, props_i_glb, n_i, labels_i = compute_semantic_vs_global_by_obj_ratio(
    df_imnet,
    key_global=KEY_G_IM,
    key_obj=KEY_O_IM,
    tag_obj="fix_nonmask",
    tag_global="global",
    n_bins=10,
    require_total=150528,
    require_full_k=True
)

# ============================================================
# Plot: single figure, two panels (NO bottom (a)/(b) captions)
# ============================================================
fig, (axL, axR) = plt.subplots(
    1, 2,
    figsize=(14.8, 4.8),
    sharey=True,
    constrained_layout=True
)

draw_grouped_stacked_sem_vs_global(
    axL, props_c_sem, props_c_glb, n_c, labels_c,
    xlabel="Object ratio",
    title="CIFAR-100",
    hatch_glb="///",
    show_pct_on_sem=True,
    pct_thresh=0.06,
    show_n=True
)
axL.set_ylabel("Proportion of instances")

draw_grouped_stacked_sem_vs_global(
    axR, props_i_sem, props_i_glb, n_i, labels_i,
    xlabel="Object ratio",
    title="ImageNet",
    hatch_glb="///",
    show_pct_on_sem=True,
    pct_thresh=0.06,
    show_n=True
)

# Legends outside the right panel
handles_results = [
    mpl.patches.Patch(
        facecolor=OUTCOME_STYLE[k]["facecolor"],
        edgecolor="black",
        hatch=OUTCOME_STYLE[k]["hatch"],
        label=k,
    )
    for k in ORDER
]
handles_methods = [
    mpl.patches.Patch(facecolor="white", edgecolor="black", label=CSI_LABEL, hatch=""),
    mpl.patches.Patch(facecolor="white", edgecolor="black", label=BASELINE_LABEL, hatch="///"),
]
leg1 = axR.legend(
    handles=handles_results,
    title="Outcome",
    loc="upper left",
    bbox_to_anchor=(1.02, 1.00),
    frameon=False,
)
axR.add_artist(leg1)
axR.legend(
    handles=handles_methods,
    title="Method",
    loc="upper left",
    bbox_to_anchor=(1.02, 0.58),
    frameon=False,
)

# Save (vector)
out_pdf = PLOT_DIR / "cifar100_imagenet_object_ratio.pdf"
fig.savefig(out_pdf, bbox_inches="tight")
print("Saved:", out_pdf)

plt.show()


'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


Saved: /Users/zd3504phd/Desktop/XAIV/plots/5.4.2_ratio/cifar100_imagenet_object_ratio.pdf


/var/folders/fr/3yg6c3f55w58dtlsnjcqjmth0000gq/T/ipykernel_20202/3137826069.py:352: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
